Import Libraries and Configure Logging

In [1]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from gensim.corpora import Dictionary
from gensim.models import LdaModel, CoherenceModel
from gensim.models.phrases import Phraser
from tqdm import tqdm
import logging
import warnings
from scipy.stats import entropy

# Setup logging
logging.basicConfig(
    filename='standard_lda_inference.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logging.getLogger('gensim').setLevel(logging.WARNING)

# Download NLTK data
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

True

Initialize NLP Tools and Stopwords

In [3]:
# Initialize lemmatizer and stopwords
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# Add custom stopwords (same as training)
custom_stopwords = {
    'allow', 'class', 'available', 'part', 'case', 'lead', 'shall', 'product', 'operate',
    'operational', 'result', 'input', 'dependent', 'preference', 'item', 'without', 'let',
    'returned', 'message', 'every', 'system', 'run', 'fully', 'major', 'reasonable',
    'software', 'user', 'able', 'ability', 'support', 'year', 'expected', 'must',
    'information', 'data', 'use', 'using', 'provide', 'successfully', 'one', 'waiter',
    'include', 'accommodate', 'event', 'technique', 'recent', 'administrator', 'search',
    'add', 'allows', 'achieve', 'way', 'outside', 'release', 'launch', 'allowed', 'entered',
    'within', 'first', 'new', 'izogn', 'wcs', 'course', 'time', 'help', 'learn', 'ccr', 'cma',
    'review', 'star', 'rating', 'good', 'great', 'course', 'learn', 'learning',
    'would', 'like', 'could', 'one', 'bit', 'week', 'think', 'much', 'really',
    'lot', 'new', 'thank', 'thanks', 'many', 'well', 'also', 'get', 'time',
    'truly', 'even', 'make', 'see', 'content', 'material', 'class', 'work',
    'way', 'understand', 'information', 'helpful', 'useful', 'knowledge',
    'day', 'help', 'easy'
}
stop_words.update(custom_stopwords)

Load Seed Words (for Verification)

In [6]:
def load_seed_words_from_csv(file_path):
    """Load seed words from CSV file and return as dictionary"""
    try:
        df = pd.read_csv(file_path)
        seed_words = df.groupby('Category')['SeedWord'].apply(list).to_dict()
        logging.info(f"Successfully loaded seed words from {file_path}")
        return seed_words
    except FileNotFoundError:
        logging.error(f"Seed words file not found at {file_path}")
        raise
    except Exception as e:
        logging.error(f"Error loading seed words: {e}")
        raise

# Load seed words and update stopwords
seed_words_path = '../../../datasets/seed_words.csv'
seed_words = load_seed_words_from_csv(seed_words_path)
seed_word_set = set(word for words in seed_words.values() for word in words)
stop_words = stop_words - seed_word_set

print("Seed words loaded successfully!")
print(f"Number of seed word categories: {len(seed_words)}")

Seed words loaded successfully!
Number of seed word categories: 7


Define Preprocessing Function

In [7]:
def preprocess(text, stop_words, lemmatizer, bigram_phraser):
    """Preprocess text for LDA model"""
    if not isinstance(text, str) or not text.strip():
        return []
    
    # Convert to lowercase and remove non-alphabetic characters
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Tokenize and lemmatize
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(word) for word in tokens 
              if word not in stop_words and len(word) > 2]
    
    # Apply bigram phraser
    tokens = bigram_phraser[tokens]
    return tokens

Define Coherence Evaluation Function

In [8]:
def compute_coherence_range_robust(lda_model, tokenized_reviews, dictionary, id_to_topic, window_size=50, topn_range=(10, 60, 10)):
    """
    Compute coherence scores for multiple topn values with robust error handling
    """
    logging.info("Started computing C_v coherence scores for multiple topn values")
    print("\n--- Computing C_v Coherence for Multiple topn Values (Robust) ---")
    
    start, end, step = topn_range
    coherence_results = []
    
    # Pre-filter topics to avoid problematic ones
    valid_topics = []
    for topic_id in range(lda_model.num_topics):
        topic_words = lda_model.show_topic(topic_id, topn=20)
        if len(topic_words) > 0 and topic_words[0][1] > 0.01:
            valid_topics.append(topic_id)
    
    print(f"Valid topics for coherence evaluation: {valid_topics}")
    
    for topn in range(start, end + 1, step):
        logging.info(f"Computing C_v coherence for topn={topn}")
        print(f"\nComputing C_v Coherence for topn={topn}")
        
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                
                coherence_model = CoherenceModel(
                    model=lda_model,
                    texts=tokenized_reviews,
                    dictionary=dictionary,
                    coherence='c_v',
                    topn=min(topn, len(dictionary) // 2),
                    window_size=window_size
                )
                
                per_topic_coherence = coherence_model.get_coherence_per_topic()
                
                valid_coherences = []
                print(f"\nPer-Topic C_v Scores (topn={topn}):")
                for i, score in enumerate(per_topic_coherence):
                    topic_name = id_to_topic[i]
                    if np.isnan(score):
                        logging.warning(f"Topic #{i} ({topic_name}): C_v Score = NaN (skipped)")
                        print(f"Topic #{i}: {topic_name} - C_v Score = NaN (skipped)")
                    else:
                        valid_coherences.append(score)
                        logging.info(f"Topic #{i} ({topic_name}): C_v Score = {score:.4f} (topn={topn})")
                        print(f"Topic #{i}: {topic_name} - C_v Score = {score:.4f}")
                
                if valid_coherences:
                    overall_coherence = np.mean(valid_coherences)
                    logging.info(f"Overall C_v Coherence Score (topn={topn}): {overall_coherence:.4f}")
                    print(f"\nOverall C_v Coherence Score (topn={topn}): {overall_coherence:.4f}")
                    print(f"Valid topics: {len(valid_coherences)}/{len(per_topic_coherence)}")
                else:
                    overall_coherence = np.nan
                    logging.warning(f"No valid coherence scores for topn={topn}")
                    print(f"\nNo valid coherence scores for topn={topn}")
                
                print("-" * 50)
                
                coherence_results.append({
                    'topn': topn,
                    'per_topic_coherence': per_topic_coherence,
                    'overall_coherence': overall_coherence,
                    'valid_topics': len(valid_coherences),
                    'total_topics': len(per_topic_coherence)
                })
                
        except Exception as e:
            logging.error(f"Error computing coherence for topn={topn}: {e}")
            print(f"Error computing coherence for topn={topn}: {e}")
            continue
    
    logging.info("Completed computing C_v coherence scores for multiple topn values")
    print("\n--- Completed Computing C_v Coherence for Multiple topn Values ---")
    return coherence_results

Analyze Coherence Results

In [9]:
def analyze_coherence_results(coherence_results):
    """
    Analyze and visualize coherence results
    """
    print("\n=== COHERENCE ANALYSIS SUMMARY ===")
    
    summary_data = []
    for result in coherence_results:
        summary_data.append({
            'topn': result['topn'],
            'overall_coherence': result['overall_coherence'],
            'valid_topics': result['valid_topics'],
            'total_topics': result['total_topics'],
            'success_rate': result['valid_topics'] / result['total_topics']
        })
    
    summary_df = pd.DataFrame(summary_data)
    
    print("\nCoherence Summary:")
    print(summary_df.to_string(index=False, float_format='%.4f'))
    
    valid_results = summary_df[~summary_df['overall_coherence'].isna()]
    if not valid_results.empty:
        best_topn = valid_results.loc[valid_results['overall_coherence'].idxmax(), 'topn']
        best_coherence = valid_results['overall_coherence'].max()
        print(f"\nBest coherence: {best_coherence:.4f} at topn={best_topn}")
    
    return summary_df

Set Input Parameters

In [15]:
class Args:
    input = '../../../datasets/review_test.csv'

args = Args()
print(f"Input file: {args.input}")

Input file: ../../../datasets/review_test.csv


Load Pre-trained Models

In [13]:
logging.info("Loading saved models")
try:
    lda_model = LdaModel.load('../train/models/standard_lda_model')
    dictionary = Dictionary.load('../train/models/standard_dictionary')
    bigram_phraser = Phraser.load('../train/models/standard_bigram_phraser')
    logging.info("Loaded LDA model, dictionary, and bigram model")
    print("✓ Successfully loaded all models")
    print(f"  - Number of topics: {lda_model.num_topics}")
    print(f"  - Dictionary size: {len(dictionary)}")
except FileNotFoundError as e:
    logging.error(f"Failed to load model files: {e}")
    print(f"\nError: Failed to load model files: {e}")
    raise

✓ Successfully loaded all models
  - Number of topics: 7
  - Dictionary size: 15791


Load and Prepare Dataset

In [16]:
logging.info("Loading new dataset")
try:
    df = pd.read_csv(args.input)
    df = df[['processed_reviews']].copy()
    df['processed_reviews'] = df['processed_reviews'].fillna("")
    logging.info("Loaded new dataset")
    print(f"✓ Successfully loaded dataset with {len(df)} reviews")
except FileNotFoundError as e:
    logging.error(f"Failed to load dataset {args.input}: {e}")
    print(f"\nError: Failed to load dataset {args.input}: {e}")
    raise

✓ Successfully loaded dataset with 71630 reviews


Preprocess Reviews

In [17]:
logging.info("Preprocessing new dataset")
tqdm.pandas()
tokenized_reviews = df['processed_reviews'].progress_apply(
    lambda x: preprocess(x, stop_words, lemmatizer, bigram_phraser)
)

valid_indices = [i for i, tokens in enumerate(tokenized_reviews) if tokens]
tokenized_reviews = [tokens for tokens in tokenized_reviews if tokens]

if not tokenized_reviews:
    raise ValueError("No valid reviews after preprocessing.")

filtered_df = df.iloc[valid_indices].copy()
logging.info(f"Filtered to {len(filtered_df)} valid reviews")
print(f"✓ Preprocessed {len(tokenized_reviews)} valid reviews")

100%|██████████| 71630/71630 [00:05<00:00, 12922.92it/s]

✓ Preprocessed 70412 valid reviews


Create Bag-of-Words Corpus

In [18]:
logging.info("Creating BoW corpus")
corpus = [dictionary.doc2bow(text) for text in tokenized_reviews]
logging.info("Created BoW corpus")
print(f"✓ Created corpus with {len(corpus)} documents")

✓ Created corpus with 70412 documents


Infer Topics

In [19]:
logging.info("Inferring topics")
topic_matrix = np.zeros((len(corpus), lda_model.num_topics))

for i, doc in enumerate(corpus):
    topics = lda_model.get_document_topics(doc, minimum_probability=0.0)
    for topic_id, prob in topics:
        topic_matrix[i, topic_id] = prob

topic_assignments = np.argmax(topic_matrix, axis=1)
logging.info("Completed topic inference")
print(f"✓ Completed topic inference for {len(corpus)} documents")

✓ Completed topic inference for 70412 documents


Define Topic Mappings

In [20]:
topic_name_to_id = {f"Topic_{i}": i for i in range(lda_model.num_topics)}
id_to_topic = {v: k for k, v in topic_name_to_id.items()}

print("\nTopic Name to Integer ID Mapping:")
for name, id in topic_name_to_id.items():
    print(f"  {name}: {id}")


Topic Name to Integer ID Mapping:
  Topic_0: 0
  Topic_1: 1
  Topic_2: 2
  Topic_3: 3
  Topic_4: 4
  Topic_5: 5
  Topic_6: 6


Compute Coherence Scores

In [21]:
coherence_results = compute_coherence_range_robust(
    lda_model=lda_model,
    tokenized_reviews=tokenized_reviews,
    dictionary=dictionary,
    id_to_topic=id_to_topic,
    window_size=50,
    topn_range=(10, 60, 10)
)

summary_df = analyze_coherence_results(coherence_results)


--- Computing C_v Coherence for Multiple topn Values (Robust) ---
Valid topics for coherence evaluation: [0, 1, 2, 3, 4, 5, 6]

Computing C_v Coherence for topn=10

Per-Topic C_v Scores (topn=10):
Topic #0: Topic_0 - C_v Score = 0.4714
Topic #1: Topic_1 - C_v Score = 0.4902
Topic #2: Topic_2 - C_v Score = 0.5134
Topic #3: Topic_3 - C_v Score = 0.7128
Topic #4: Topic_4 - C_v Score = 0.4893
Topic #5: Topic_5 - C_v Score = 0.3984
Topic #6: Topic_6 - C_v Score = 0.2854

Overall C_v Coherence Score (topn=10): 0.4801
Valid topics: 7/7
--------------------------------------------------

Computing C_v Coherence for topn=20

Per-Topic C_v Scores (topn=20):
Topic #0: Topic_0 - C_v Score = 0.4628
Topic #1: Topic_1 - C_v Score = 0.4454
Topic #2: Topic_2 - C_v Score = 0.4062
Topic #3: Topic_3 - C_v Score = 0.7278
Topic #4: Topic_4 - C_v Score = 0.3980
Topic #5: Topic_5 - C_v Score = 0.3619
Topic #6: Topic_6 - C_v Score = 0.2181

Overall C_v Coherence Score (topn=20): 0.4314
Valid topics: 7/7
-----

Verify Seed Word Probabilities

In [22]:
logging.info("Verifying Seed Word Probabilities and Ranks in Standard LDA Model")
print("\n--- Verifying Seed Word Probabilities and Ranks in Standard LDA Model ---")

seed_word_ids = {
    topic: [dictionary.token2id[word] for word in words if word in dictionary.token2id]
    for topic, words in seed_words.items()
}

for topic_name, words in seed_words.items():
    topic_id = topic_name_to_id[f"Topic_{min(range(lda_model.num_topics), key=lambda x: x)}"]  # Map to Topic_0 if no direct mapping
    print(f"\nSeed Words: {topic_name} (Mapped to ID: {topic_id})")
    logging.info(f"Seed Words: {topic_name} (Mapped to ID: {topic_id})")
    
    for word in words:
        if word in dictionary.token2id:
            word_id = dictionary.token2id[word]
            term_topics = lda_model.get_term_topics(word_id, minimum_probability=0.0)
            print(f"  - Seed Word '{word}'")
            logging.info(f"  - Seed Word '{word}'")
            
            for t_id, prob in term_topics:
                t_name = id_to_topic[t_id]
                topic_words_probs = lda_model.show_topic(t_id, topn=len(dictionary))
                word_to_rank = {w: idx + 1 for idx, (w, _) in enumerate(topic_words_probs)}
                rank = word_to_rank.get(word, "N/A")
                log_message = f"    * Topic {t_name} (ID: {t_id}): Probability = {prob:.4f}, Rank = {rank}"
                logging.info(log_message)
                print(log_message)
        else:
            logging.warning(f"Seed Word '{word}' for topic '{topic_name}' was not in the final dictionary.")
            print(f"  - '{word}' (Not in dictionary)")

print("----------------------------------------------------------\n")


--- Verifying Seed Word Probabilities and Ranks in Standard LDA Model ---

Seed Words: F (Mapped to ID: 0)
  - Seed Word 'player'
    * Topic Topic_3 (ID: 3): Probability = 0.0000, Rank = 1578
  - Seed Word 'display'
    * Topic Topic_5 (ID: 5): Probability = 0.0002, Rank = 414
  - Seed Word 'meeting'
    * Topic Topic_0 (ID: 0): Probability = 0.0001, Rank = 668
    * Topic Topic_1 (ID: 1): Probability = 0.0001, Rank = 848
    * Topic Topic_3 (ID: 3): Probability = 0.0000, Rank = 1945
  - Seed Word 'dispute'
    * Topic Topic_1 (ID: 1): Probability = 0.0001, Rank = 992
  - Seed Word 'program'
    * Topic Topic_0 (ID: 0): Probability = 0.0043, Rank = 50
    * Topic Topic_2 (ID: 2): Probability = 0.0009, Rank = 184
    * Topic Topic_3 (ID: 3): Probability = 0.0009, Rank = 283
    * Topic Topic_4 (ID: 4): Probability = 0.0007, Rank = 329
    * Topic Topic_5 (ID: 5): Probability = 0.0015, Rank = 127
  - Seed Word 'clinical'
    * Topic Topic_4 (ID: 4): Probability = 0.0015, Rank = 156
  -

Assign Topics to Reviews and Save

In [23]:
logging.info("Started getting topic distributions")
all_topics_data = []
entropies = [entropy(probs) for probs in topic_matrix]
logging.info(f"Average topic entropy: {np.mean(entropies):.4f}, Std: {np.std(entropies):.4f}")
print(f"\nAverage topic entropy: {np.mean(entropies):.4f}, Std: {np.std(entropies):.4f}")

for i, (review_text, topic_probs) in enumerate(zip(filtered_df['processed_reviews'], topic_matrix)):
    dominant_topic = np.argmax(topic_probs)
    dominant_prob = topic_probs[dominant_topic]
    topic_name = id_to_topic[dominant_topic]
    review_entropy = entropy(topic_probs)
    
    all_topics_data.append({
        'original_index': valid_indices[i],
        'processed_reviews': review_text,
        'topic': dominant_topic,
        'topic_name': topic_name,
        'confidence': dominant_prob,
        'topic_probs': ','.join(map(str, topic_probs)),
        'entropy': review_entropy
    })

all_topics_df = pd.DataFrame(all_topics_data)
all_topics_path = '../../datasets/standard_lda_test_reviews_with_topic.csv'
all_topics_df.to_csv(all_topics_path, index=False)
logging.info(f"All reviews with topic assignments saved to {all_topics_path}")
print(f"\nAll reviews with topic assignments saved to {all_topics_path}")

logging.info("Sample of saved data:")
logging.info(all_topics_df[['processed_reviews', 'topic', 'topic_name', 'confidence', 'entropy']].head(2).to_string())
print("\nSample of saved data:")
print(all_topics_df[['processed_reviews', 'topic', 'topic_name', 'confidence', 'entropy']].head(2))


Average topic entropy: 0.6172, Std: 0.4440


OSError: Cannot save file into a non-existent directory: '..\..\datasets'